# Alternative Sentiment Scores

## Problem Definition

**Question.** Do the locally stored article-level sentiment scores form a valid observed feature table without model downloads?

**Role in the workflow.** Validate the timestamped sentiment input used to create news events.

**Inputs.** Local news and local FinBERT-score Parquets.

**Outputs.** A score-consistency summary and the validated sentiment-feature path; no model is downloaded or rerun.

**Why this method.** Reusing the stored scores guarantees an offline clean run and preserves the already processed observed text.

**Assumptions.** Scores inherit the language model and news-vendor biases; only the publication-time score is used downstream.

**Handoff.** The sentiment path to `event_labeling.ipynb`.


## Real Data Check and Preprocessing

The score table is checked against the observed news ids. No return, label, or holdout outcome is joined during this validation.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]
news_path = PROJECT_ROOT / "data/research_data/alternative/data/aapl_2025-01-01_2025-12-31.parquet"
score_path = PROJECT_ROOT / "data/research_data/alternative/features/aapl_finbert_sentiment_scores_2025-01-01_2025-12-31.parquet"

news = pd.read_parquet(news_path)
scores = pd.read_parquet(score_path)
probability_columns = ["sentiment_positive", "sentiment_negative", "sentiment_neutral"]

scores["created_at"] = pd.to_datetime(scores["created_at"], utc=True)
assert scores["id"].isin(news["id"]).all()
assert scores[probability_columns].apply(lambda column: column.between(0, 1).all()).all()
assert np.allclose(scores[probability_columns].sum(axis=1), 1.0, atol=1e-4)
assert np.allclose(
    scores["sentiment_score"],
    scores["sentiment_positive"] - scores["sentiment_negative"],
    atol=1e-6,
)

summary = scores[[*probability_columns, "sentiment_score"]].describe().T
summary.insert(0, "rows", len(scores))
display(summary)
display(scores[["created_at", "headline", "sentiment_score"]].head())


## Results, Limitations, and Handoff

Not every news row has a retained score because preprocessing filtered unusable text/URL records. This missingness can affect representativeness and is reported rather than filled with synthetic sentiment.

The next notebook receives the timestamped observed sentiment table. No conclusion in this notebook is evidence of live-trading profitability.
